In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
 
print("=" * 60)
print("SYNTHETIC DATA GENERATION - IMPROVED VERSION")
print("=" * 60)

SYNTHETIC DATA GENERATION - IMPROVED VERSION


In [3]:
# IMPORTANT: Time-based cutoff to prevent data leakage
observation_cutoff = pd.to_datetime('2023-09-30')  # Features up to here
reference_date = pd.to_datetime('2023-12-31')      # Final date
 
print(f"\n⏰ TIME WINDOWS:")
print(f"   Observation cutoff: {observation_cutoff.strftime('%Y-%m-%d')}")
print(f"   Reference date: {reference_date.strftime('%Y-%m-%d')}")
print(f"   Prediction window: 3 months (Oct-Dec 2023)")


⏰ TIME WINDOWS:
   Observation cutoff: 2023-09-30
   Reference date: 2023-12-31
   Prediction window: 3 months (Oct-Dec 2023)


In [4]:
# STEP 1: Load cleaned data
print("\n[1/5] Loading cleaned customer data...")
df = pd.read_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/processed/telco_cleaned.csv')
print(f"✓ Loaded {len(df)} customers")
 
# Calculate signup dates
signup_dates = reference_date - pd.to_timedelta(df['tenure'] * 30, unit='D')


[1/5] Loading cleaned customer data...
✓ Loaded 7043 customers


In [5]:
# STEP 2: Generate TRANSACTIONS with improved realism
# ============================================================
print("\n[2/5] Generating transaction records...")
 
transactions = []
txn_id = 1
 
for idx, row in df.iterrows():
    customer_id = row['customerID']
    tenure = row['tenure']
    monthly_charge = row['MonthlyCharges']
    payment_method = row['PaymentMethod']
    is_churned = (row['Churn'] == 'Yes')
    signup_date = signup_dates.iloc[idx]
    
    # Generate transactions only up to observation_cutoff
    # (prevents using future transactions as features)
    max_date = min(signup_date + timedelta(days=int(tenure * 30)), observation_cutoff)
    
    # Calculate months to generate
    months_to_generate = int((max_date - signup_date).days / 30)
    
    for month in range(months_to_generate):
        txn_date = signup_date + pd.DateOffset(months=month)
        
        # Only generate if before observation cutoff
        if txn_date > observation_cutoff:
            break
        
        # Add realistic noise to amount (±5%)
        amount = round(monthly_charge * (1 + np.random.uniform(-0.05, 0.05)), 2)
        
        # Payment status logic (churned customers have more failures)
        # IMPROVED: Calculate days until churn for better targeting
        days_until_churn = (max_date - txn_date).days if is_churned else 999
        
        if is_churned and days_until_churn < 90:
            # Last 3 months: 15% failure rate
            payment_status = np.random.choice(['paid', 'failed'], p=[0.85, 0.15])
        else:
            # Normal: 2% failure rate
            payment_status = np.random.choice(['paid', 'failed'], p=[0.98, 0.02])
        
        # IMPROVED: Failed payments = $0 revenue (realistic!)
        if payment_status == 'failed':
            amount = 0.0
        
        transactions.append({
            'transaction_id': f'TXN{txn_id:08d}',
            'customer_id': customer_id,
            'transaction_date': txn_date.strftime('%Y-%m-%d'),
            'amount': amount,
            'payment_method': payment_method,
            'payment_status': payment_status
        })
        txn_id += 1
    
    if (idx + 1) % 1000 == 0:
        print(f"   Processed {idx + 1}/{len(df)} customers...")
 
transactions_df = pd.DataFrame(transactions)
print(f"✓ Generated {len(transactions_df):,} transactions")
print(f"   Failed payments: {(transactions_df['payment_status']=='failed').sum():,}")
print(f"   Failed payment revenue: ${(transactions_df[transactions_df['payment_status']=='failed']['amount']).sum():.2f}")


[2/5] Generating transaction records...
   Processed 1000/7043 customers...
   Processed 2000/7043 customers...
   Processed 3000/7043 customers...
   Processed 4000/7043 customers...
   Processed 5000/7043 customers...
   Processed 6000/7043 customers...
   Processed 7000/7043 customers...
✓ Generated 202,377 transactions
   Failed payments: 4,370
   Failed payment revenue: $0.00


In [6]:
# STEP 3: Generate SESSIONS with churn behavior drift
# ============================================================
print("\n[3/5] Generating session records...")
 
internet_customers = df[df['InternetService'] != 'No'].copy()
print(f"   {len(internet_customers)} customers have internet")
 
sessions = []
session_id = 1
 
for idx, row in internet_customers.iterrows():
    customer_id = row['customerID']
    tenure = row['tenure']
    internet_type = row['InternetService']
    is_churned = (row['Churn'] == 'Yes')
    signup_date = signup_dates.loc[idx]
    
    # Base session frequency
    if internet_type == 'Fiber optic':
        base_sessions_per_month = np.random.randint(8, 12)
    else:
        base_sessions_per_month = np.random.randint(5, 8)
    
    # Generate sessions only up to observation cutoff
    max_date = min(signup_date + timedelta(days=int(tenure * 30)), observation_cutoff)
    total_days = int((max_date - signup_date).days)
    
    current_date = signup_date
    
    while current_date < max_date:
        # Days until churn (for behavior drift)
        days_until_churn = (max_date - current_date).days
        
        # IMPROVED: Declining activity for churners in last 90 days
        if is_churned and days_until_churn < 90:
            # Linear decay from 100% to 30% activity
            activity_multiplier = max(0.3, days_until_churn / 90)
        else:
            # Normal or growing activity
            activity_multiplier = np.random.uniform(0.9, 1.1)
        
        sessions_this_month = int(base_sessions_per_month * activity_multiplier)
        
        # Generate sessions for this month
        for _ in range(sessions_this_month):
            session_start = current_date + timedelta(days=np.random.randint(0, 30))
            
            # Don't exceed observation cutoff
            if session_start > observation_cutoff:
                break
            
            # Session duration (also affected by churn proximity)
            if internet_type == 'Fiber optic':
                base_duration = 40
            else:
                base_duration = 20
            
            # IMPROVED: Shorter sessions for churners near churn date
            if is_churned and days_until_churn < 90:
                duration_multiplier = max(0.5, days_until_churn / 90)
            else:
                duration_multiplier = 1.0
            
            duration_minutes = max(1, int(np.random.normal(base_duration, 10) * duration_multiplier))
            
            # Pages viewed correlates with duration
            pages_viewed = max(1, int(duration_minutes / 3 * np.random.uniform(0.8, 1.2)))
            
            device = np.random.choice(['mobile', 'desktop', 'tablet'], p=[0.6, 0.3, 0.1])
            
            sessions.append({
                'session_id': f'SES{session_id:09d}',
                'customer_id': customer_id,
                'session_start': session_start.strftime('%Y-%m-%d %H:%M:%S'),
                'duration_minutes': duration_minutes,
                'pages_viewed': pages_viewed,
                'device': device
            })
            session_id += 1
        
        current_date += timedelta(days=30)
    
    if session_id % 100000 == 0:
        print(f"   Generated {session_id:,} sessions...")
 
sessions_df = pd.DataFrame(sessions)
print(f"✓ Generated {len(sessions_df):,} sessions")
print(f"   Avg duration: {sessions_df['duration_minutes'].mean():.1f} minutes")


[3/5] Generating session records...
   5517 customers have internet
✓ Generated 1,218,789 sessions
   Avg duration: 32.9 minutes


In [7]:
# STEP 4: Generate SUPPORT TICKETS
# ============================================================
print("\n[4/5] Generating support tickets...")
 
tickets = []
ticket_id = 1
 
for idx, row in df.iterrows():
    customer_id = row['customerID']
    tenure = row['tenure']
    has_tech_support = (row['TechSupport'] == 'Yes')
    is_churned = (row['Churn'] == 'Yes')
    signup_date = signup_dates.iloc[idx]
    
    # Base ticket rate
    if not has_tech_support and row['InternetService'] != 'No':
        tickets_per_year = np.random.uniform(3, 5)
    elif has_tech_support:
        tickets_per_year = np.random.uniform(1, 2)
    else:
        tickets_per_year = np.random.uniform(0.5, 1.5)
    
    # Calculate max date (observation cutoff)
    max_date = min(signup_date + timedelta(days=int(tenure * 30)), observation_cutoff)
    years = (max_date - signup_date).days / 365
    expected_tickets = int(tickets_per_year * years)
    
    # IMPROVED: Churned customers spike in last 60 days
    if is_churned:
        expected_tickets += np.random.randint(2, 5)
    
    for _ in range(expected_tickets):
        # Random date during tenure
        days_range = int((max_date - signup_date).days)
        days_offset = np.random.randint(0, max(1, days_range))
        ticket_date = signup_date + timedelta(days=days_offset)
        
        # Don't exceed observation cutoff
        if ticket_date > observation_cutoff:
            continue
        
        # IMPROVED: Bias tickets toward end for churners
        days_until_churn = (max_date - ticket_date).days
        if is_churned and np.random.random() < 0.5 and days_until_churn < 60:
            # 50% of churner tickets in last 60 days
            ticket_date = max_date - timedelta(days=np.random.randint(0, 60))
        
        # Category
        if is_churned:
            category = np.random.choice(['billing', 'technical', 'service', 'account'],
                                       p=[0.35, 0.30, 0.25, 0.10])
        else:
            category = np.random.choice(['billing', 'technical', 'service', 'account'],
                                       p=[0.25, 0.40, 0.20, 0.15])
        
        resolution_hours = np.random.randint(2, 48)
        resolved_date = ticket_date + timedelta(hours=resolution_hours)
        
        # Satisfaction (churned = less satisfied)
        if is_churned:
            satisfaction = np.random.choice([1, 2, 3, 4, 5], p=[0.3, 0.3, 0.2, 0.15, 0.05])
        else:
            satisfaction = np.random.choice([1, 2, 3, 4, 5], p=[0.10, 0.15, 0.25, 0.30, 0.20])
        
        tickets.append({
            'ticket_id': f'TKT{ticket_id:08d}',
            'customer_id': customer_id,
            'created_date': ticket_date.strftime('%Y-%m-%d %H:%M:%S'),
            'resolved_date': resolved_date.strftime('%Y-%m-%d %H:%M:%S'),
            'category': category,
            'resolution_hours': resolution_hours,
            'satisfaction_score': satisfaction
        })
        ticket_id += 1
 
tickets_df = pd.DataFrame(tickets)
print(f"✓ Generated {len(tickets_df):,} support tickets")
print(f"   Avg satisfaction: {tickets_df['satisfaction_score'].mean():.2f}/5.0")



[4/5] Generating support tickets...
✓ Generated 41,190 support tickets
   Avg satisfaction: 3.09/5.0


In [8]:
# STEP 5: VALIDATION (NEW - Critical Quality Check!)
# ============================================================
print("\n[5/5] Validating data quality...")
 
# Check 1: Referential integrity
print("   Checking referential integrity...")
assert transactions_df['customer_id'].isin(df['customerID']).all(), "Orphaned transactions!"
assert sessions_df['customer_id'].isin(df['customerID']).all(), "Orphaned sessions!"
assert tickets_df['customer_id'].isin(df['customerID']).all(), "Orphaned tickets!"
print("   All customer_ids valid (no orphaned records)")
 
# Check 2: No future data leakage
print("   Checking for time leakage...")
txn_max_date = pd.to_datetime(transactions_df['transaction_date']).max()
sess_max_date = pd.to_datetime(sessions_df['session_start']).max()
tkt_max_date = pd.to_datetime(tickets_df['created_date']).max()
 
assert txn_max_date <= observation_cutoff, f"Transactions after cutoff: {txn_max_date}"
assert sess_max_date <= observation_cutoff, f"Sessions after cutoff: {sess_max_date}"
assert tkt_max_date <= observation_cutoff, f"Tickets after cutoff: {tkt_max_date}"
print(f"   All data before observation cutoff ({observation_cutoff.strftime('%Y-%m-%d')})")
 
# Check 3: Failed payments have $0 amount
failed_txns = transactions_df[transactions_df['payment_status'] == 'failed']
assert (failed_txns['amount'] == 0).all(), "Failed payments should have $0 amount!"
print(f"   All {len(failed_txns):,} failed payments have $0 revenue")
 
# Check 4: Data volume sanity checks
assert len(transactions_df) > 0, "No transactions generated!"
assert len(sessions_df) > 0, "No sessions generated!"
assert len(tickets_df) > 0, "No tickets generated!"
print(f"   All tables have data")
 
print("✓ All validation checks passed!")
 
# Save tables
print("\n[6/6] Saving synthetic tables...")
 
transactions_df.to_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/transactions.csv', index=False)
print(f"✓ Saved transactions.csv ({len(transactions_df):,} rows)")
 
sessions_df.to_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/sessions.csv', index=False)
print(f"✓ Saved sessions.csv ({len(sessions_df):,} rows)")
 
tickets_df.to_csv(r'/Users/siddharthm/Desktop/churn-prediction/data/synthetic/support_tickets.csv', index=False)
print(f"✓ Saved support_tickets.csv ({len(tickets_df):,} rows)")
 
# Summary
print("\n" + "=" * 60)
print("GENERATION COMPLETE - IMPROVED VERSION")
print("=" * 60)
print(f"\nTotal records: {len(transactions_df) + len(sessions_df) + len(tickets_df):,}")
print(f"\nKey improvements applied:")
print(f"  Time-based cutoff (no data after {observation_cutoff.strftime('%Y-%m-%d')})")
print(f"  Churn behavior drift (declining activity in last 90 days)")
print(f"  Failed payments = $0 revenue")
print(f"  Data validation checks")
print("\n✓ Dataset ready for feature engineering!")
print("=" * 60)


[5/5] Validating data quality...
   Checking referential integrity...
   All customer_ids valid (no orphaned records)
   Checking for time leakage...
   All data before observation cutoff (2023-09-30)
   All 4,370 failed payments have $0 revenue
   All tables have data
✓ All validation checks passed!

[6/6] Saving synthetic tables...
✓ Saved transactions.csv (202,377 rows)
✓ Saved sessions.csv (1,218,789 rows)
✓ Saved support_tickets.csv (41,190 rows)

GENERATION COMPLETE - IMPROVED VERSION

Total records: 1,462,356

Key improvements applied:
  Time-based cutoff (no data after 2023-09-30)
  Churn behavior drift (declining activity in last 90 days)
  Failed payments = $0 revenue
  Data validation checks

✓ Dataset ready for feature engineering!
